# Laboratório de Ciência de Dados
## Aula 9 — Detecção de anomalias I: comportamento esperado e métodos estatísticos

Este notebook é autoinstrucional. Leia as explicações, execute as células na ordem e responda às questões propostas ao longo da atividade.

Em muitos problemas de Ciência de Dados, o objetivo não é apenas prever uma classe. Também precisamos reconhecer observações que se afastam do comportamento normalmente observado. Uma transação bancária muito diferente das demais, um sensor industrial que apresenta uma leitura incompatível com seu histórico ou um acesso a sistema realizado de maneira incomum podem ser exemplos de anomalias.

Nesta primeira aula, o foco será compreender **como definir um comportamento esperado** e como detectar desvios utilizando métodos relativamente simples e interpretáveis.

### Objetivos

Ao final da atividade, você deverá ser capaz de:

- distinguir anomalia, valor extremo e erro de dados;
- identificar anomalias univariadas com IQR e escore robusto;
- reconhecer anomalias multivariadas;
- utilizar distância de Mahalanobis;
- utilizar PCA para detectar observações com alto erro de reconstrução;
- compreender o papel do limiar de decisão;
- comparar diferentes detectores em uma base simulada.

Na Aula 10, trabalharemos com algoritmos desenvolvidos especificamente para detecção de anomalias.

# 1. Preparação do ambiente

Utilizaremos `numpy`, `pandas`, `matplotlib`, `scikit-learn` e `scipy`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import precision_score, recall_score, f1_score
from scipy.spatial.distance import mahalanobis
from scipy.stats import chi2

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# 2. O que significa uma observação anômala?

Uma observação anômala é suficientemente diferente do padrão utilizado como referência. Isso não significa, por si só, que a observação esteja errada.

Considere três casos:

- uma compra de R\$ 9.000 feita por um cliente que normalmente gasta R$ 50;
- uma temperatura de 500 °C registrada por um sensor cuja faixa física termina em 120 °C;
- um cliente de alta renda que realiza uma compra cara, mas compatível com seu histórico.

No primeiro caso pode existir um evento raro e verdadeiro. No segundo, há forte indício de erro de medição. No terceiro, o valor é extremo para a população, mas talvez não seja anômalo para aquele cliente.

A detecção de anomalias indica observações que merecem investigação. A decisão sobre o significado do desvio depende do contexto.

### Questão 1

Uma observação detectada como anômala deve ser removida automaticamente da base? Explique.

# 3. Tipos de anomalia

Nem todo desvio aparece da mesma forma.

Uma **anomalia pontual** é incomum quando comparada diretamente ao restante da população, como uma transação de valor excepcionalmente alto.

Uma **anomalia contextual** depende da situação. Uma compra de R$ 800 pode ser comum para determinado cliente e estranha para outro; uma temperatura de 30 °C pode ser normal à tarde e incomum durante uma madrugada de inverno.

Uma **anomalia coletiva** aparece quando um conjunto ou uma sequência apresenta comportamento estranho, mesmo que cada observação isolada pareça plausível. Uma sequência incomum de acessos a um sistema é um exemplo.

Nesta aula trabalharemos principalmente com anomalias pontuais e multivariadas. A distinção contextual é importante porque, em aplicações reais, o comportamento esperado pode depender do usuário, do tempo ou de outras condições.

# 4. Base didática de transações

A maior parte das transações será gerada segundo um padrão comum. Uma pequena parcela seguirá outro processo e funcionará como anomalia conhecida apenas para fins de avaliação.

Os detectores não utilizarão a coluna `anomalia_real` durante o ajuste.

In [ ]:
n_normais = 1200
n_anomalias = 60

normais = pd.DataFrame({
    "valor": rng.lognormal(mean=4.8, sigma=0.55, size=n_normais),
    "transacoes_1h": np.clip(rng.poisson(1.1, n_normais), 0, 6),
    "distancia_habitual_km": np.clip(rng.exponential(8, n_normais), 0, 60),
    "hora": np.clip(rng.normal(15, 4, n_normais), 0, 23),
    "anomalia_real": 0
})

anomalias = pd.DataFrame({
    "valor": rng.lognormal(mean=7.0, sigma=0.65, size=n_anomalias),
    "transacoes_1h": rng.integers(4, 11, size=n_anomalias),
    "distancia_habitual_km": rng.uniform(60, 350, size=n_anomalias),
    "hora": rng.choice([0, 1, 2, 3, 4, 23], size=n_anomalias),
    "anomalia_real": 1
})

df = pd.concat([normais, anomalias], ignore_index=True)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

df["valor"] = df["valor"].round(2)
df["distancia_habitual_km"] = df["distancia_habitual_km"].round(2)
df["hora"] = df["hora"].round(1)

df.head()

In [ ]:
print("Número de observações:", len(df))
print("Anomalias simuladas:", int(df["anomalia_real"].sum()))
print("Percentual de anomalias:", round(100 * df["anomalia_real"].mean(), 2), "%")

df.drop(columns="anomalia_real").describe().round(2)

# 5. Inspeção visual

Uma inspeção simples não substitui um detector, mas ajuda a formular hipóteses sobre o comportamento dos dados.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["valor"], bins=50)
plt.xlabel("Valor da transação")
plt.ylabel("Frequência")
plt.title("Distribuição dos valores das transações")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(
    df["valor"],
    df["distancia_habitual_km"],
    alpha=0.55
)
plt.xlabel("Valor da transação")
plt.ylabel("Distância do local habitual (km)")
plt.title("Valor × distância")
plt.grid(alpha=0.2)
plt.show()

### Questão 2

Existem observações que não parecem extremas em `valor`, mas passam a chamar atenção quando `valor` e `distancia_habitual_km` são analisados conjuntamente?

# 6. Intervalo interquartil — IQR

Uma forma simples de detectar valores extremos em uma variável é utilizar os quartis.

$$
IQR = Q_3 - Q_1
$$

Podemos sinalizar valores abaixo de

$$
Q_1 - 1{,}5 \times IQR
$$

ou acima de

$$
Q_3 + 1{,}5 \times IQR
$$

O fator 1,5 é uma convenção comum, não uma regra universal.

In [ ]:
q1 = df["valor"].quantile(0.25)
q3 = df["valor"].quantile(0.75)
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

df["anomalia_iqr"] = (
    (df["valor"] < limite_inferior)
    | (df["valor"] > limite_superior)
).astype(int)

print("Q1:", round(q1, 2))
print("Q3:", round(q3, 2))
print("IQR:", round(iqr, 2))
print("Limite superior:", round(limite_superior, 2))
print("Sinalizadas:", int(df["anomalia_iqr"].sum()))

O método analisa somente uma variável. Uma transação de valor comum realizada a centenas de quilômetros do local habitual não será detectada por esse critério se o valor estiver dentro da faixa usual.

# 7. Escore robusto baseado em mediana e MAD

O Z-score tradicional utiliza média e desvio padrão, medidas sensíveis a valores extremos. Uma alternativa robusta utiliza a mediana e o **MAD — Median Absolute Deviation**.

$$
MAD = mediana(|x_i - mediana(x)|)
$$

Um escore robusto bastante utilizado é:

$$
z_i^* =
0{,}6745
\frac{x_i - mediana(x)}
{MAD}
$$

Valores com grande magnitude estão mais afastados do centro robusto da distribuição.

In [ ]:
def modified_z_score(valores):
    valores = np.asarray(valores)
    mediana = np.median(valores)
    mad = np.median(np.abs(valores - mediana))

    if mad == 0:
        return np.zeros(len(valores))

    return 0.6745 * (valores - mediana) / mad


df["z_robusto_valor"] = modified_z_score(df["valor"])

# 3.5 é uma referência comum, mas deve ser tratado como parâmetro.
df["anomalia_z_robusto"] = (
    np.abs(df["z_robusto_valor"]) > 3.5
).astype(int)

print("Sinalizadas:", int(df["anomalia_z_robusto"].sum()))

df.sort_values(
    "z_robusto_valor",
    ascending=False
)[["valor", "z_robusto_valor", "anomalia_real"]].head(10)

### Questão 3

IQR e escore robusto sinalizaram exatamente o mesmo conjunto de observações? Por que dois métodos univariados podem divergir?

# 8. Anomalias multivariadas

Uma combinação de valores pode ser incomum mesmo quando nenhum atributo isolado é extremo. Para analisar esse tipo de situação, os atributos precisam ser considerados conjuntamente.

Antes disso, vamos padronizá-los, pois estão em escalas bastante diferentes.

In [ ]:
atributos = [
    "valor",
    "transacoes_1h",
    "distancia_habitual_km",
    "hora"
]

X = df[atributos]

scaler = StandardScaler()
X_std = scaler.fit_transform(X)

pd.DataFrame(
    X_std,
    columns=atributos
).agg(["mean", "std"]).round(3)

# 9. Distância de Mahalanobis

A distância de Mahalanobis mede afastamento levando em conta a covariância entre as variáveis.

$$
D_M(x) =
\sqrt{
(x-\mu)^T
\Sigma^{-1}
(x-\mu)
}
$$

O vetor $\mu$ representa o centro e $\Sigma$ a matriz de covariância.

In [ ]:
media = X_std.mean(axis=0)
cov = np.cov(X_std, rowvar=False)
cov_inv = np.linalg.pinv(cov)

distancias_mahal = np.array([
    mahalanobis(linha, media, cov_inv)
    for linha in X_std
])

df["mahalanobis"] = distancias_mahal

df[
    atributos + ["mahalanobis", "anomalia_real"]
].sort_values(
    "mahalanobis",
    ascending=False
).head(10)

## 9.1 Definindo um limiar

Sob hipóteses específicas, o quadrado da distância de Mahalanobis pode ser comparado com uma distribuição qui-quadrado. Utilizaremos o percentil de 99% como referência didática.

In [ ]:
p = len(atributos)

limite_chi2 = chi2.ppf(
    0.99,
    df=p
)

df["anomalia_mahal"] = (
    df["mahalanobis"] ** 2 > limite_chi2
).astype(int)

print("Limiar de D²:", round(limite_chi2, 3))
print("Sinalizadas:", int(df["anomalia_mahal"].sum()))

### Questão 4

Por que uma observação pode ter distância de Mahalanobis elevada mesmo sem possuir o maior valor em nenhuma das variáveis?

# 10. PCA e erro de reconstrução

O PCA pode ser utilizado além da visualização e redução de dimensionalidade.

Ao manter apenas as principais componentes, construímos uma representação aproximada dos dados. Observações compatíveis com a estrutura dominante tendem a ser reconstruídas razoavelmente bem. Observações pouco compatíveis podem produzir erro de reconstrução elevado.

In [ ]:
pca = PCA(
    n_components=2,
    random_state=RANDOM_STATE
)

X_reduzido = pca.fit_transform(X_std)
X_reconstruido = pca.inverse_transform(X_reduzido)

erro_reconstrucao = np.mean(
    (X_std - X_reconstruido) ** 2,
    axis=1
)

df["erro_pca"] = erro_reconstrucao

print(
    "Variância preservada:",
    round(100 * pca.explained_variance_ratio_.sum(), 2),
    "%"
)

df[
    atributos + ["erro_pca", "anomalia_real"]
].sort_values(
    "erro_pca",
    ascending=False
).head(10)

## 10.1 Limiar baseado em percentil

Utilizaremos o percentil de 95% do erro de reconstrução. Isso implica, aproximadamente, sinalizar os 5% maiores erros.

In [ ]:
limiar_pca = np.quantile(
    df["erro_pca"],
    0.95
)

df["anomalia_pca"] = (
    df["erro_pca"] > limiar_pca
).astype(int)

print("Limiar:", round(limiar_pca, 4))
print("Sinalizadas:", int(df["anomalia_pca"].sum()))

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["erro_pca"], bins=50)
plt.axvline(limiar_pca, linestyle="--")
plt.xlabel("Erro de reconstrução")
plt.ylabel("Frequência")
plt.title("Distribuição do erro de reconstrução")
plt.show()

### Questão 5

Se o limiar passar do percentil 95% para o percentil 99%, a quantidade de alertas tende a aumentar ou diminuir? O que isso representa em termos de sensibilidade?

# 11. Comparação dos detectores

Como a base foi simulada, conhecemos as anomalias reais. Assim, podemos calcular precisão, recall e F1. Em uma aplicação não supervisionada real, esses rótulos frequentemente não existem.

In [ ]:
def avaliar_detector(nome, coluna):
    y_real = df["anomalia_real"]
    y_pred = df[coluna]

    return {
        "metodo": nome,
        "sinalizadas": int(y_pred.sum()),
        "precisao": precision_score(y_real, y_pred, zero_division=0),
        "recall": recall_score(y_real, y_pred, zero_division=0),
        "f1": f1_score(y_real, y_pred, zero_division=0)
    }


resultados = [
    avaliar_detector("IQR - valor", "anomalia_iqr"),
    avaliar_detector("Z robusto - valor", "anomalia_z_robusto"),
    avaliar_detector("Mahalanobis", "anomalia_mahal"),
    avaliar_detector("PCA - reconstrução", "anomalia_pca")
]

pd.DataFrame(resultados).round(3)

### Questão 6

Qual método obteve maior recall? Qual obteve maior precisão? O que os resultados sugerem sobre a limitação de métodos que analisam apenas uma variável?

# 12. Anomalia não é sinônimo de fraude

Mesmo quando uma transação é sinalizada, o método não está afirmando que ela seja fraudulenta. Ele está dizendo apenas que a observação se afasta do padrão utilizado como referência.

Uma transação legítima pode ser rara. Uma fraude também pode imitar o comportamento comum.

Detecção de anomalias é, portanto, frequentemente uma etapa de triagem e investigação.

# 13. Exercícios

## Exercício 1 — sensibilidade do IQR

Repita o método usando multiplicadores 1,0; 2,0 e 3,0 no lugar de 1,5. Para cada configuração, informe quantidade sinalizada, precisão, recall e F1.

In [ ]:
# Complete aqui.

## Exercício 2 — outro atributo

Aplique o escore robusto a `distancia_habitual_km` e compare as observações detectadas com aquelas sinalizadas pelo valor da transação.

In [ ]:
# Complete aqui.

## Exercício 3 — limiar de Mahalanobis

Repita a análise usando os percentis 0,95; 0,975; 0,99 e 0,995 da distribuição qui-quadrado. Compare precisão e recall.

In [ ]:
# Complete aqui.

## Exercício 4 — número de componentes no PCA

Compare `n_components=1`, `2` e `3`, sempre utilizando o percentil 95% do erro de reconstrução como limiar. Explique o efeito do número de componentes sobre o erro.

In [ ]:
# Complete aqui.

# 14. Desafio final

Selecione cinco observações com maior distância de Mahalanobis.

Para cada uma, mostre os atributos originais, a distância de Mahalanobis, o erro de reconstrução do PCA, os dois rótulos produzidos e `anomalia_real`.

Depois escreva uma interpretação curta do motivo pelo qual a observação parece incomum.

In [ ]:
# Complete aqui.

# 15. Questões de revisão

**1. Qual é a diferença entre uma anomalia e um erro de dados?**

**2. Por que um método univariado pode perder uma anomalia multivariada?**

**3. Por que padronizamos variáveis antes de calcular distâncias?**

**4. O que a distância de Mahalanobis considera além do afastamento ao centro?**

**5. Como o PCA pode ser utilizado para detecção de anomalias?**

**6. Qual é o efeito de aumentar um limiar de detecção?**

**7. Por que uma anomalia não deve ser removida automaticamente?**

**8. Por que precisão e recall podem ser impossíveis de calcular em uma aplicação real sem rótulos?**

# 16. Conclusão

Nesta aula, a detecção de anomalias foi tratada como identificação de desvios em relação ao comportamento esperado.

Métodos univariados são simples e interpretáveis, mas não capturam combinações incomuns entre atributos. A distância de Mahalanobis permite uma análise multivariada, enquanto o PCA oferece outra perspectiva: observações que não se ajustam bem à estrutura dominante tendem a apresentar maior erro de reconstrução.

Na Aula 10, ampliaremos essa análise com algoritmos específicos para detecção de anomalias.